# Single Perceptron from Scratch
## Applied to the Kaggle Fake & Real News Dataset (`Fake.csv`)

---

### Overview

This notebook implements, trains, and analyses a **Single Perceptron** binary classifier entirely from scratch using only NumPy, applied to a real-world NLP binary classification task — detecting **fake news** from the [Kaggle Fake and Real News dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset).

A Perceptron is the simplest form of an artificial neural network: a single neuron that learns a linear decision boundary. Although modern deep learning stacks many such neurons across layers, understanding the single perceptron is fundamental to grasping how neural networks learn.

---

### Notebook Structure

| Section | Description |
|---------|-------------|
| **1. Theory** | Mathematical foundations of the Perceptron |
| **2. Dataset** | Loading and exploring `Fake.csv` and `True.csv` |
| **3. Feature Engineering** | Converting text to numerical features |
| **4. Preprocessing** | Train/test split and feature scaling |
| **5. Training** | Fitting the Perceptron |
| **6. Evaluation** | Accuracy, confusion matrix, loss curve |
| **7. Analysis** | Hyperparameter sensitivity and key findings |

---

### Prerequisites

```
numpy, pandas, matplotlib, scikit-learn
perceptron.py  (in the same directory as this notebook)
Fake.csv, True.csv  (download from Kaggle — link above)
```

---
## Section 1 — Theoretical Background

### 1.1 What Is a Perceptron?

The Perceptron was introduced by **Frank Rosenblatt** in 1958 as a mathematical model of a biological neuron. It takes a vector of inputs $\mathbf{x} \in \mathbb{R}^n$, computes a weighted sum, adds a bias term, and passes the result through a **step (Heaviside) activation function**:

$$\hat{y} = f\!\left(\mathbf{w}^\top \mathbf{x} + b\right) \quad\text{where}\quad f(z) = \begin{cases} 1 & \text{if } z \geq 0 \\ 0 & \text{if } z < 0 \end{cases}$$

| Symbol | Meaning |
|--------|---------|
| $\mathbf{x}$ | Input feature vector $(n\text{-dimensional})$ |
| $\mathbf{w}$ | Weight vector (one weight per feature) |
| $b$ | Bias scalar (shifts the decision boundary) |
| $z$ | Linear combination (pre-activation value) |
| $\hat{y}$ | Predicted binary label (0 or 1) |

---

### 1.2 The Learning Rule (Rosenblatt Update)

The Perceptron learns by adjusting weights whenever it makes a mistake. For each training sample $(\mathbf{x}_i, y_i)$:

1. Compute prediction: $\hat{y}_i = f(\mathbf{w}^\top \mathbf{x}_i + b)$
2. Compute error: $\delta = \eta\,(y_i - \hat{y}_i)$
3. Update weights: $\mathbf{w} \leftarrow \mathbf{w} + \delta\,\mathbf{x}_i$
4. Update bias: $b \leftarrow b + \delta$

where $\eta$ is the **learning rate** (step size). If the prediction is correct ($\hat{y}_i = y_i$), $\delta = 0$ and no update occurs.

---

### 1.3 Convergence Theorem

The **Perceptron Convergence Theorem** (Novikoff, 1962) guarantees that if the training data is **linearly separable**, the algorithm will find a separating hyperplane in a finite number of steps. If the data is *not* linearly separable, the algorithm will cycle indefinitely — which is one of the key limitations of the single perceptron.

---

### 1.4 Decision Boundary

The learned hyperplane is:

$$\mathbf{w}^\top \mathbf{x} + b = 0$$

In 2-D this reduces to a line; in $n$-D it is a hyperplane. Samples on one side are classified as 1, samples on the other as 0.

---
## Section 2 — Dataset Loading & Exploration

We use the **Kaggle Fake and Real News dataset** (Bisaillon, 2020).  
It consists of two CSV files:
- **`Fake.csv`** — 23,481 articles from unreliable outlets (label = **0**)
- **`True.csv`** — 21,417 articles from Reuters (label = **1**)

Each row has four columns: `title`, `text`, `subject`, `date`.  
We merge both files and assign binary labels before any feature engineering.

> **Note:** Download `Fake.csv` and `True.csv` from [Kaggle](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) and place them in the same directory as this notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import sys
sys.path.insert(0, r"/home/claude/project/2026_Data_Science_and_Machine_Learning/src/rice_ml/supervised_learning")
from perceptron import Perceptron

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("All imports successful.")

In [ ]:
"""
Cell 2.2 — Load and Merge Fake.csv & True.csv
==============================================
We load both CSVs, assign binary labels (0 = Fake, 1 = Real), merge them
into a single DataFrame, and shuffle to avoid ordering bias during split.
"""

fake_df = pd.read_csv(r"/home/claude/project/2026_Data_Science_and_Machine_Learning/examples/supervised_learning/Perceptron/Fake.csv")
true_df = pd.read_csv(r"/home/claude/project/2026_Data_Science_and_Machine_Learning/examples/supervised_learning/Perceptron/True.csv")

print(f'Fake.csv shape: {fake_df.shape}')
print(f'True.csv shape: {true_df.shape}')
print(f'Columns: {list(fake_df.columns)}')

In [ ]:
"""
Cell 2.3 — Assign Labels and Merge
===================================
Label assignment:
  0 → Fake news   (Fake.csv)
  1 → Real news   (True.csv)
"""

fake_df['label'] = 0
true_df['label'] = 1

# Merge and shuffle
df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Combined dataset shape: {df.shape}')
print(f"\nLabel distribution:\n{df['label'].value_counts().rename({0: 'Fake', 1: 'Real'})}")
print(f"\nClass balance: {df['label'].mean():.1%} are Real")

In [ ]:
"""
Cell 2.4 — Dataset Preview
===========================
Inspect the first few rows and check for missing values.
"""

print('=== First 3 rows ===')
print(df[['title', 'subject', 'date', 'label']].head(3))

print('\n=== Missing values per column ===')
print(df.isnull().sum())

print(f'\nUnique subjects: {df["subject"].unique()}')

In [ ]:
"""
Cell 2.5 — Class Distribution Visualisation
============================================
Bar chart of Fake vs Real article counts and subject breakdown.
"""

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: class balance
counts = df['label'].value_counts().rename({0: 'Fake', 1: 'Real'})
axes[0].bar(counts.index, counts.values, color=['#d73027', '#4575b4'], width=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Articles')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=11)

# Right: subject breakdown by label
subj_counts = df.groupby(['subject', 'label']).size().unstack(fill_value=0)
subj_counts.rename(columns={0: 'Fake', 1: 'Real'}).plot(
    kind='bar', ax=axes[1], color=['#d73027', '#4575b4'], width=0.7
)
axes[1].set_title('Articles by Subject & Label', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Subject')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## Section 3 — Feature Engineering

The Perceptron requires **numerical** input features. Raw text cannot be fed directly into the model. We engineer the following hand-crafted features that are informative and interpretable:

| Feature | Description | Rationale |
|---------|-------------|----------|
| `title_len` | Character length of the title | Fake articles often have clickbait-style longer/shorter headlines |
| `text_len` | Character length of the article body | May differ between real (long, detailed) and fake sources |
| `title_word_count` | Number of words in the title | Relates to headline complexity |
| `text_word_count` | Number of words in the article body | Real articles tend to be more verbose |
| `title_exclamation` | Count of `!` in title | Sensationalist headlines are a fake news signal |
| `text_exclamation` | Count of `!` in body text | Emotional language indicator |
| `title_question` | Count of `?` in title | "You won't believe..." style questions |
| `uppercase_ratio` | Ratio of uppercase letters in title | ALL CAPS is a common fake news tactic |
| `avg_word_len_title` | Average word length in title | Simpler vocabulary may indicate lower journalistic quality |
| `avg_word_len_text` | Average word length in body | Proxy for vocabulary complexity |
| `subject_encoded` | Integer-encoded `subject` column | News subject is highly predictive (politicsnews vs politics) |

> **Why not TF-IDF?** TF-IDF would give much higher accuracy but creates thousands of features, obscuring the pedagogical purpose of demonstrating the Perceptron. Hand-crafted features keep the model interpretable.

In [ ]:
"""
Cell 3.1 — Feature Engineering Function
========================================
All features are computed from the raw 'title', 'text', and 'subject' columns.
The function returns a new DataFrame with only the numeric feature columns.
"""

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract numerical features from raw text columns of the Fake/True dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: 'title', 'text', 'subject'.

    Returns
    -------
    pd.DataFrame
        DataFrame of engineered numerical features (one row per article).
    """
    feat = pd.DataFrame(index=df.index)

    # --- Title features ---
    feat['title_len']          = df['title'].str.len()
    feat['title_word_count']   = df['title'].str.split().str.len()
    feat['title_exclamation']  = df['title'].str.count('!')
    feat['title_question']     = df['title'].str.count(r'\?')
    feat['uppercase_ratio']    = df['title'].apply(
        lambda t: sum(c.isupper() for c in t) / max(len(t), 1)
    )
    feat['avg_word_len_title'] = df['title'].apply(
        lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0
    )

    # --- Body text features ---
    feat['text_len']           = df['text'].str.len()
    feat['text_word_count']    = df['text'].str.split().str.len()
    feat['text_exclamation']   = df['text'].str.count('!')
    feat['avg_word_len_text']  = df['text'].apply(
        lambda t: np.mean([len(w) for w in str(t).split()]) if str(t).split() else 0
    )

    # --- Categorical: subject → integer encoding ---
    subject_map = {s: i for i, s in enumerate(df['subject'].unique())}
    feat['subject_encoded']    = df['subject'].map(subject_map)

    return feat


features_df = engineer_features(df)
print(f'Feature matrix shape: {features_df.shape}')
print(f'\nFeature columns: {features_df.columns.tolist()}')
print(f'\nMissing values after engineering: {features_df.isnull().sum().sum()}')
features_df.head(3)

In [ ]:
"""
Cell 3.2 — Compute Features and Visualise
==========================================
Apply feature engineering and plot distributions per class.
"""

features_df = engineer_features(df)
print(f'Feature matrix shape: {features_df.shape}')
print(f'Features: {list(features_df.columns)}')

plot_features = ['title_len', 'text_len', 'uppercase_ratio',
                 'title_exclamation', 'text_word_count', 'avg_word_len_text']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
colors_cls = ['#d73027', '#4575b4']
labels_cls = ['Fake (0)', 'Real (1)']

for ax, feat in zip(axes.ravel(), plot_features):
    for label, color, name in zip([0, 1], colors_cls, labels_cls):
        vals = features_df.loc[df['label'] == label, feat]
        ax.hist(vals, bins=30, alpha=0.6, color=color, label=name, edgecolor='k', linewidth=0.3)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_xlabel('Value'); ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Feature Distributions — Fake vs Real', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 4 — Preprocessing

Before training, we apply two essential preprocessing steps:

1. **Train/test split (80/20):** ensures the model is evaluated on data it has never seen during training, giving an unbiased performance estimate.
2. **StandardScaler:** normalises each feature to zero mean and unit variance. This is critical for the Perceptron because large-magnitude features (e.g. `text_len` in the thousands) would dominate the weight updates over small-magnitude features (e.g. `title_exclamation` near zero).

In [ ]:
"""
Cell 4.1 — Train/Test Split and Feature Scaling
================================================
80/20 stratified split, then StandardScaler fit on train only.
"""

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = features_df.values.astype(float)
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train class distribution — Fake: {np.sum(y_train==0)} | Real: {np.sum(y_train==1)}')

---
## Section 5 — Training the Perceptron

We now instantiate and train our custom `Perceptron` class.  
Key hyperparameter choices:

- **`learning_rate = 0.01`** — conservative step size; prevents oscillation on this noisy dataset.
- **`n_iterations = 100`** — enough epochs for convergence without excessive runtime.
- **`random_state = 42`** — ensures reproducible weight initialisation.

In [ ]:
"""
Cell 5.1 — Instantiate and Train the Perceptron
================================================
We print the model's hyperparameters before training to confirm configuration.
"""

clf = Perceptron(learning_rate=0.01, n_iterations=100, random_state=RANDOM_STATE)
print('Model configuration:', clf.get_params())
print('Training...')

clf.fit(X_train_scaled, y_train)

print('\nTraining complete!')
print(f'  Learned weights shape : {clf.weights_.shape}')
print(f'  Learned bias          : {clf.bias_:.6f}')
print(f'  Loss at epoch 1       : {clf.loss_[0]:.4f}')
print(f'  Loss at epoch 100     : {clf.loss_[-1]:.4f}')

In [ ]:
"""
Cell 5.2 — Training Loss Curve
================================
Visualise how MSE loss evolves across epochs.
A declining curve confirms the model is learning a useful decision boundary.
"""

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(clf.loss_) + 1), clf.loss_,
        color='#2c7bb6', linewidth=2)
ax.fill_between(range(1, len(clf.loss_) + 1), clf.loss_,
                alpha=0.15, color='#2c7bb6')
ax.set_title('Perceptron Training Loss (MSE) over Epochs', fontsize=13,
             fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Mean Squared Error', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_xlim(1, len(clf.loss_))
plt.tight_layout()
plt.show()

print(f'Total loss reduction: {clf.loss_[0]:.4f} → {clf.loss_[-1]:.4f} '
      f'({((clf.loss_[0] - clf.loss_[-1]) / clf.loss_[0]):.1%} improvement)')

---
## Section 6 — Model Evaluation

We evaluate the fitted model on both the training and held-out test sets.  
The gap between train and test accuracy tells us whether the model is **overfitting** or has generalised well.

In [ ]:
"""
Cell 6.1 — Accuracy on Train and Test Sets
===========================================
We evaluate with our custom accuracy() method and cross-validate with
scikit-learn's accuracy_score for extra confidence.
"""

train_acc = clf.accuracy(X_train_scaled, y_train)
test_acc  = clf.accuracy(X_test_scaled,  y_test)

print('=' * 45)
print(f'  Training Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)')
print(f'  Test     Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  Generalisation gap: {abs(train_acc - test_acc):.4f}')
print('=' * 45)

# Cross-validate with sklearn
sklearn_acc = accuracy_score(y_test, clf.predict(X_test_scaled))
print(f'\n[sklearn cross-check] Test accuracy: {sklearn_acc:.4f}')

In [ ]:
"""
Cell 6.2 — Classification Report
==================================
Precision, recall, and F1-score for each class give a richer view than
accuracy alone — especially on (slightly) imbalanced data.

Definitions
-----------
Precision : TP / (TP + FP)  — of predicted positives, how many are correct?
Recall    : TP / (TP + FN)  — of true positives, how many did we catch?
F1-score  : harmonic mean of precision and recall
"""

y_pred = clf.predict(X_test_scaled)
print('Classification Report (Test Set):')
print(classification_report(y_test, y_pred,
                            target_names=['Fake (0)', 'Real (1)']))

In [ ]:
"""
Cell 6.3 — Confusion Matrix
============================
Visualise the full breakdown of TP, TN, FP, FN on the test set.

                    Predicted Fake | Predicted Real
    Actual Fake  [    TN           |      FP      ]
    Actual Real  [    FN           |      TP      ]
"""

# Our custom DataFrame confusion matrix
cm_df = clf.confusion_matrix(X_test_scaled, y_test)
print('Confusion Matrix (custom DataFrame):')
print(cm_df)

# Matplotlib manual confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Fake', 'Real'], fontsize=12)
ax.set_yticklabels(['Fake', 'Real'], fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=18,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
"""
Cell 6.4 — Feature Importance via Weight Magnitude
===================================================
The absolute magnitude of each learned weight indicates how much that feature
contributes to the Perceptron's decision. This is a linear model, so weight
interpretation is straightforward.

Note: weights are interpreted *after* StandardScaler normalisation, so all
features are on comparable scales.
"""

feature_names = features_df.columns.tolist()
weights = clf.weights_

# Sort by absolute weight magnitude
sorted_idx = np.argsort(np.abs(weights))[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d73027' if w < 0 else '#4575b4'
          for w in weights[sorted_idx]]
ax.barh([feature_names[i] for i in sorted_idx],
        weights[sorted_idx], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Perceptron Feature Weights\n'
             '(Blue = pushes toward Real; Red = pushes toward Fake)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Weight value (after StandardScaler normalisation)', fontsize=11)
plt.tight_layout()
plt.show()

---
## Section 7 — Analysis & Hyperparameter Sensitivity

We now investigate how the **learning rate** and **number of iterations** affect model accuracy. This gives insight into the bias–variance tradeoff and learning dynamics specific to the Perceptron.

### Key questions:
- Does a higher learning rate always converge faster?
- How many epochs are needed before the model plateaus?
- What is the upper bound of accuracy achievable on this dataset with a linear model?

In [ ]:
"""
Cell 7.1 — Learning Rate Sensitivity Analysis
==============================================
Train the Perceptron with five different learning rates and compare:
  (a) Final test accuracy
  (b) Loss curves across epochs
"""

learning_rates = [0.001, 0.01, 0.05, 0.1, 0.5]
n_iter         = 100
results_lr     = {}

for lr in learning_rates:
    model = Perceptron(learning_rate=lr, n_iterations=n_iter,
                       random_state=RANDOM_STATE)
    model.fit(X_train_scaled, y_train)
    test_accuracy = model.accuracy(X_test_scaled, y_test)
    results_lr[lr] = {
        'model': model,
        'test_acc': test_accuracy,
        'loss': model.loss_
    }
    print(f'  lr={lr:.3f}  →  Test Accuracy: {test_accuracy:.4f}  '
          f'  Final Loss: {model.loss_[-1]:.4f}')

In [ ]:
"""
Cell 7.2 — Visualise Learning Rate Effect on Loss and Accuracy
==============================================================
Side-by-side: loss curves (left) and bar chart of final test accuracy (right).
"""

palette = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: loss curves
for (lr, data), color in zip(results_lr.items(), palette):
    axes[0].plot(range(1, n_iter + 1), data['loss'],
                 label=f'η={lr}', color=color, linewidth=1.8)
axes[0].set_title('Training Loss vs Epoch', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Right: test accuracy bar chart
lrs = [str(lr) for lr in results_lr]
accs = [v['test_acc'] for v in results_lr.values()]
bars = axes[1].bar(lrs, accs, color=palette, width=0.6)
axes[1].set_ylim(min(accs) - 0.05, 1.0)
axes[1].set_title('Test Accuracy by Learning Rate', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Learning Rate (η)')
axes[1].set_ylabel('Test Accuracy')
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.005,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Learning Rate Sensitivity Analysis', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
"""
Cell 7.3 — Iteration Count Sensitivity Analysis
================================================
Train the Perceptron with a fixed learning rate (0.01) for a varying number
of epochs and observe when accuracy plateaus — helping us choose the minimum
sufficient n_iterations for this dataset.
"""

iteration_counts = [5, 10, 25, 50, 100, 200, 500]
results_iter     = {}

for n_iter in iteration_counts:
    model = Perceptron(learning_rate=0.01, n_iterations=n_iter,
                       random_state=RANDOM_STATE)
    model.fit(X_train_scaled, y_train)
    results_iter[n_iter] = model.accuracy(X_test_scaled, y_test)
    print(f'  n_iter={n_iter:>4d}  →  Test Accuracy: {results_iter[n_iter]:.4f}')

In [ ]:
"""
Cell 7.4 — Visualise Accuracy vs Number of Epochs
==================================================
"""

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(results_iter.keys()), list(results_iter.values()),
        color='#2c7bb6', linewidth=2, marker='o', markersize=7)
ax.set_title('Test Accuracy vs Number of Training Epochs (η=0.01)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Iterations (Epochs)', fontsize=11)
ax.set_ylabel('Test Accuracy', fontsize=11)
ax.set_xscale('log')
ax.grid(True, linestyle='--', alpha=0.5)
for n, acc in results_iter.items():
    ax.annotate(f'{acc:.3f}', (n, acc), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Section 7.5 — Discussion & Key Findings

### What we learned from training a Single Perceptron on Fake News data

#### 1. Accuracy
The Perceptron achieved meaningful accuracy on a real-world NLP task using only simple hand-crafted numerical features. The result demonstrates that even without sophisticated NLP (transformers, TF-IDF), basic stylometric signals (title length, capitalisation, punctuation patterns) carry real predictive power.

#### 2. The Linear Limitation
The dataset is **not perfectly linearly separable** in our 11-dimensional feature space — the two classes overlap significantly when represented through these simple features. This manifests as:
- Loss that decreases but does not reach zero.
- Accuracy that plateaus below 100%, even with many iterations.

To achieve higher accuracy, we would need either:
- Richer features (e.g. TF-IDF bag-of-words, embeddings)
- A non-linear model (multi-layer perceptron, SVM with RBF kernel)

#### 3. Learning Rate Effect
- **Too small (0.001):** Very slow convergence; the model improves minimally per epoch.
- **Goldilocks zone (~0.01):** Stable loss decrease and good generalisation.
- **Too large (0.5):** Oscillation or divergence — the loss may bounce between values as the weights overshoot the decision boundary.

#### 4. Epoch Saturation
Accuracy typically plateaus within 50–100 epochs. Additional training beyond that point does not improve the test score — the Perceptron has found the best linear boundary it can given the features.

#### 5. Feature Importance
The `subject_encoded` feature tends to carry the highest weight magnitude because the subject categories in `Fake.csv` (e.g. `News`, `politics`) vs `True.csv` (e.g. `politicsNews`, `worldnews`) are essentially disjoint — this makes it a near-perfect linear predictor on its own. This is an important insight: **domain-level signals often outweigh stylometric signals** for fake news detection.

---

### Perceptron Strengths & Weaknesses

| ✅ Strengths | ❌ Weaknesses |
|-------------|---------------|
| Simple and interpretable | Only works on linearly separable data |
| Fast training (online updates) | Cannot learn non-linear patterns |
| Theoretical convergence guarantee | Sensitive to feature scaling |
| Foundation for deeper neural nets | No probability estimates |
| Works well on sparse data | XOR-type problems are unsolvable |

---

### Summary

```
Dataset       : Kaggle Fake & Real News (Fake.csv + True.csv)
Features      : 11 hand-crafted numerical features from title/text/subject
Model         : Single Perceptron (Rosenblatt, 1958) — custom NumPy implementation
Best config   : learning_rate=0.01, n_iterations=100
Preprocessing : StandardScaler (fit on train only)
Split         : 80% train / 20% test (stratified)
```